# Session 1: spread the weight, or the ground gives way
**A. Real-world problem.** A stiletto heel can dent a wooden floor, yet a heavy building
stands on soil without sinking. The force is large, so the building spreads it over a
wide *footing*. The idea is one division: pressure = force / area.

**Plan.** (1) compute pressure, (2) compare heel, shoe and snowshoe, (3) find how wide a
square footing must be so the soil is not overloaded, (4) plot pressure against width.
Educational model only: never use it to design a real foundation.

In [ ]:
import math
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

# Typical textbook allowable bearing pressures in kPa (kilonewtons per square metre).
# They are rough ranges for teaching, NOT values for design.
SOIL_RANGES = {
    "soft clay": (50.0, 100.0),
    "firm clay": (100.0, 200.0),
    "dense sand or gravel": (200.0, 400.0),
}

## Step 1: pressure = force / area
With force in kilonewtons (kN) and area in square metres (m^2) the pressure comes out in
kilopascals (kPa). 1 kPa = 1 kN per m^2.

Worked by hand: 700 N is 0.7 kN. A heel of 1 cm^2 is 0.0001 m^2, so the pressure is
0.7 / 0.0001 = 7000 kPa. A snowshoe pair of 0.2 m^2 gives 0.7 / 0.2 = 3.5 kPa.

In [ ]:
def pressure(force_kn, area_m2):
    """Pressure in kPa from a force in kN spread over an area in m^2."""
    if np.any(np.asarray(area_m2) <= 0):
        raise ValueError("area must be positive")
    return force_kn / area_m2


def person_pressures(weight_n=700.0):
    """Pressure (kPa) under a stiletto heel, a shoe and a pair of snowshoes (illustrative areas)."""
    areas = {"stiletto heel (1 cm^2)": 1e-4, "shoe (300 cm^2)": 0.03, "snowshoes (2000 cm^2)": 0.2}
    return {name: pressure(weight_n / 1000.0, a) for name, a in areas.items()}

## Step 2: a square footing under a column
A column carries a load P (kN) into a square footing of side B (m). The area is B * B.
At this level we ignore the weight of the footing itself (Session 2 adds it).

In [ ]:
def footing_pressure(load_kn, side_m):
    """Average pressure (kPa) under a square footing."""
    return pressure(load_kn, side_m * side_m)


def is_safe(load_kn, side_m, allowable_kpa):
    """True when the pressure does not exceed the allowable bearing pressure (tiny tolerance for round-off)."""
    return footing_pressure(load_kn, side_m) <= allowable_kpa * (1 + 1e-12)


def round_up(x, step=0.05):
    """Round x up to a multiple of step (footings are built in neat sizes)."""
    return math.ceil(x / step - 1e-9) * step


def min_side(load_kn, allowable_kpa):
    """Smallest square side (m) so that load / side^2 <= allowable: B = sqrt(P / q)."""
    return math.sqrt(load_kn / allowable_kpa)

## Step 3: see it
Left: pressure falls as the footing widens (area grows with B squared). The soil bands show
typical allowable ranges. Right: heel, shoe and snowshoe on a log scale.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    for name, v in person_pressures().items():
        print(f"{name:26s} -> {v:10.1f} kPa")
    print("300 kN on 1.5 m square:", round(footing_pressure(300, 1.5), 1), "kPa")
    print("smallest side for 75 kPa:", round(min_side(300, 75), 3), "m")

    B = np.linspace(0.8, 4.0, 200)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(B, footing_pressure(300, B), color="C0", label="pressure under 300 kN column")
    for (name, (lo, hi)), col in zip(SOIL_RANGES.items(), ("C3", "C1", "C2")):
        ax.axhspan(lo, hi, color=col, alpha=0.15, label=f"{name} ({lo:.0f} to {hi:.0f} kPa)")
    ax.set_ylim(0, 450)
    ax.set_xlabel("footing side B (m)")
    ax.set_ylabel("pressure on soil (kPa)")
    ax.set_title("A wider footing means lower pressure (typical ranges, not for design)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(OUT / "pressure_vs_width.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6, 3.5))
    pp = person_pressures()
    ax2.barh(list(pp.keys()), list(pp.values()), color=["C3", "C1", "C2"])
    ax2.set_xscale("log")
    ax2.set_xlabel("pressure (kPa, log scale)")
    ax2.set_title("Same 700 N person, very different pressure")
    ax2.grid(alpha=0.3, axis="x")
    fig2.tight_layout()
    fig2.savefig(OUT / "heel_vs_snowshoe.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Change the load in `lesson.py` to 450 kN and print the pressure on a 1.5 m footing before you run it. Predict first.
2. **Stretch.** Write `footing_for_soil(P, soil_name)` that uses the lower end of the typical range of a soil in SOIL_RANGES and returns the side rounded up to 0.05 m.
3. **Explain.** Explain, with the words area and pressure, why a person lying flat on thin ice is safer than one standing on it.

## Assessment
- **A. Conceptual.** Why does a snowshoe stop you sinking even though your weight does not change? What does that say about a building?
- **B. Numerical.** A 450 kN column stands on a 1.5 m square footing. Find the pressure, say whether it is safe on firm clay at 150 kPa, and find the smallest safe side.
- **C. Graph and simulation interpretation.** On the pressure-against-width graph for a 300 kN column, what happens to the pressure when the width is halved? Where does the curve cross a 100 kPa soil line?
- **D. Design and optimisation challenge.** Design: a 300 kN column could stand on soft clay (75 kPa) or on dense sand (300 kPa). Find the smallest square footing for each and compare the concrete area. What other things would an engineer worry about?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("hand calculation: 600 kN on 2 m x 2 m is 150 kPa", abs(L.footing_pressure(600, 2.0) - 150.0) < 1e-9))
    pp = L.person_pressures(700.0)
    heel, shoe, snow = list(pp.values())
    checks.append(("heel is 7000 kPa and snowshoes are 3.5 kPa", abs(heel - 7000.0) < 1e-6 and abs(snow - 3.5) < 1e-9))
    checks.append(("shoe pressure is about 23.3 kPa", abs(shoe - 23.333) < 0.01))
    checks.append(("worked example: 300 kN on 1.5 m square is 133.3 kPa, too much for soft clay at 75 kPa",
                   abs(L.footing_pressure(300, 1.5) - 133.333) < 0.01 and not L.is_safe(300, 1.5, 75.0)))
    checks.append(("worked example: smallest side for 300 kN at 75 kPa is exactly 2.0 m", abs(L.min_side(300, 75) - 2.0) < 1e-12))
    checks.append(("edge: exactly 2.0 m is safe at 75 kPa (boundary, no round-off failure)", L.is_safe(300, 2.0, 75.0)))
    checks.append(("edge: 1.95 m is not safe at 75 kPa", not L.is_safe(300, 1.95, 75.0)))
    checks.append(("doubling the side divides the pressure by 4", abs(L.footing_pressure(300, 1.0) / L.footing_pressure(300, 2.0) - 4.0) < 1e-12))
    checks.append(("round_up(1.732) is 1.75 and round_up(2.0) stays 2.0", abs(L.round_up(1.732) - 1.75) < 1e-9 and abs(L.round_up(2.0) - 2.0) < 1e-9))
    try:
        L.pressure(10, 0)
        ok = False
    except ValueError:
        ok = True
    checks.append(("zero area is rejected", ok))
    checks.append(("typical ranges are ordered: soft clay below firm clay below dense sand",
                   L.SOIL_RANGES["soft clay"][1] <= L.SOIL_RANGES["firm clay"][0] and L.SOIL_RANGES["firm clay"][1] <= L.SOIL_RANGES["dense sand or gravel"][0]))
    checks.append(("assessment B: 450 kN on 1.5 m is 200 kPa, smallest side sqrt(3) = 1.732 m built 1.75 m, giving 147 kPa",
                   abs(L.footing_pressure(450, 1.5) - 200.0) < 1e-9 and abs(L.min_side(450, 150) - math.sqrt(3)) < 1e-12
                   and abs(L.round_up(L.min_side(450, 150)) - 1.75) < 1e-9 and abs(L.footing_pressure(450, 1.75) - 146.94) < 0.01))
    checks.append(("assessment C: 300 kN at 100 kPa needs 1.73 m; halving the width gives 4 times the pressure",
                   abs(L.min_side(300, 100) - 1.732) < 1e-3 and abs(L.footing_pressure(300, 0.5) / L.footing_pressure(300, 1.0) - 4.0) < 1e-12))
    checks.append(("assessment D: 300 kN needs 2.0 m (4.0 m2) on 75 kPa and 1.0 m (1.0 m2) on 300 kPa",
                   abs(L.min_side(300, 75) - 2.0) < 1e-12 and abs(L.min_side(300, 300) - 1.0) < 1e-12))
    checks.append(("app challenge: 600 kN on a 2.1 m footing is 136 kPa, safe on 150 kPa soil", L.is_safe(600, 2.1, 150.0) and abs(L.footing_pressure(600, 2.1) - 136.05) < 0.01))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
